In [1]:
import math
import copy
from dataclasses import dataclass

import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Function


In [2]:
@dataclass
class Config:
    batch_size: int = 256
    num_workers: int = 8
    pin_memory: bool = True
    persistent_workers: bool = True
    prefetch_factor: int = 4
    lr: float = 1e-3
    weight_decay: float = 1e-4
    num_epochs: int = 100
    val_split: int = 500
    train_split: int = 50000 - val_split
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


cfg = Config()
print(cfg.device)


cuda


In [3]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4, padding_mode="reflect"),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
        transforms.RandomErasing(
            p=0.25,
            scale=(0.02, 0.2),
            ratio=(0.3, 3.3),
            value="random",
        ),
    ]
)

test_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ]
)


In [4]:
train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform,
)

train_set, val_set = torch.utils.data.random_split(
    train_dataset,
    [cfg.train_split, cfg.val_split],
)

train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    persistent_workers=cfg.persistent_workers if cfg.num_workers > 0 else False,
    prefetch_factor=cfg.prefetch_factor if cfg.num_workers > 0 else None,
)

val_loader = torch.utils.data.DataLoader(
    val_set,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    persistent_workers=cfg.persistent_workers if cfg.num_workers > 0 else False,
    prefetch_factor=cfg.prefetch_factor if cfg.num_workers > 0 else None,
)

test_set = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform,
)

test_loader = torch.utils.data.DataLoader(
    test_set,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=cfg.pin_memory,
    persistent_workers=cfg.persistent_workers if cfg.num_workers > 0 else False,
    prefetch_factor=cfg.prefetch_factor if cfg.num_workers > 0 else None,
)


Files already downloaded and verified
Files already downloaded and verified


In [5]:
class BinaryQuantize(Function):
    @staticmethod
    def forward(ctx, input, k, t):
        ctx.save_for_backward(input, k, t)
        return torch.sign(input)

    @staticmethod
    def backward(ctx, grad_output):
        input, k, t = ctx.saved_tensors
        grad_input = k * t * (1.0 - torch.tanh(input * t).pow(2)) * grad_output
        return grad_input, None, None


In [6]:
class IRConv2d(nn.Conv2d):
    def __init__(self, *args, eps=1e-5, **kwargs):
        super().__init__(*args, **kwargs)
        self.eps = eps

        self.register_buffer("k", torch.tensor([10.0]))
        self.register_buffer("t", torch.tensor([0.1]))

        self.perturb_prob = 0.0
        self.perturb_seed = 1234

    def _maybe_flip_binary_weights(self, bw):
        if self.perturb_prob <= 0.0:
            return bw

        gen = torch.Generator(device=bw.device)
        gen.manual_seed(int(self.perturb_seed))
        mask = torch.rand(bw.shape, generator=gen, device=bw.device) < self.perturb_prob
        return torch.where(mask, -bw, bw)

    def forward(self, input):
        w = self.weight

        bw = w - w.view(w.size(0), -1).mean(dim=1).view(w.size(0), 1, 1, 1)
        bw = bw / (bw.view(bw.size(0), -1).std(dim=1).view(w.size(0), 1, 1, 1) + self.eps)

        mean_abs = bw.abs().view(bw.size(0), -1).mean(dim=1).clamp_min(self.eps)
        sw = torch.pow(
            torch.full_like(mean_abs, 2.0),
            torch.round(torch.log2(mean_abs)),
        ).view(w.size(0), 1, 1, 1).detach()

        bw = BinaryQuantize.apply(bw, self.k, self.t)
        bw = self._maybe_flip_binary_weights(bw)

        ba = BinaryQuantize.apply(input, self.k, self.t)

        return F.conv2d(
            ba,
            bw * sw,
            self.bias,
            self.stride,
            self.padding,
            self.dilation,
            self.groups,
        )


In [7]:
class IRLinear(nn.Linear):
    def __init__(self, *args, eps=1e-5, **kwargs):
        super().__init__(*args, **kwargs)
        self.eps = eps

        self.register_buffer("k", torch.tensor([10.0]))
        self.register_buffer("t", torch.tensor([0.1]))

        self.perturb_prob = 0.0
        self.perturb_seed = 4321

    def _maybe_flip_binary_weights(self, bw):
        if self.perturb_prob <= 0.0:
            return bw

        gen = torch.Generator(device=bw.device)
        gen.manual_seed(int(self.perturb_seed))
        mask = torch.rand(bw.shape, generator=gen, device=bw.device) < self.perturb_prob
        return torch.where(mask, -bw, bw)

    def forward(self, input):
        w = self.weight

        bw = w - w.mean(dim=1, keepdim=True)
        bw = bw / (bw.std(dim=1, keepdim=True) + self.eps)

        mean_abs = bw.abs().mean(dim=1).clamp_min(self.eps)
        sw = torch.pow(
            torch.full_like(mean_abs, 2.0),
            torch.round(torch.log2(mean_abs)),
        ).view(-1, 1).detach()

        bw = BinaryQuantize.apply(bw, self.k, self.t)
        bw = self._maybe_flip_binary_weights(bw)

        ba = BinaryQuantize.apply(input, self.k, self.t)

        return F.linear(ba, bw * sw, self.bias)


In [8]:
def update_ede_params(model, epoch, total_epochs, t_min=0.1, t_max=10.0):
    # epoch is zero-indexed.
    progress = epoch / max(total_epochs - 1, 1)
    t = t_min * (10 ** (progress * math.log10(t_max / t_min)))
    k = max(1.0 / t, 1.0)

    for module in model.modules():
        if isinstance(module, (IRConv2d, IRLinear)):
            module.k.fill_(k)
            module.t.fill_(t)

    return k, t


In [9]:
class Network(nn.Module):
    def __init__(
        self,
        input_channels=3,
        num_classes=10,
        image_size=32,
        dropout=0.25,
        n_ratio=1.0,  # kept for compatibility; IR layers do not use N_scale
    ):
        super().__init__()

        self.img_size = image_size
        self.cin = input_channels
        self.cout1 = 32
        self.cout2 = 32

        self.fcin = self.cout2 * (16 ** 2)
        self.fcout1 = 128
        self.fcout2 = 128
        self.fcout3 = num_classes

        self.n_ratio = n_ratio

        self.block1 = nn.Sequential(
            nn.Conv2d(self.cin, self.cout1, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(self.cout1),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),
        )

        self.block2 = nn.Sequential(
            IRConv2d(self.cout1, self.cout2, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(self.cout2),
            nn.ReLU(),
        )

        self.block3 = nn.Sequential(
            IRLinear(self.fcin, self.fcout1),
            nn.BatchNorm1d(self.fcout1),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.block4 = nn.Sequential(
            IRLinear(self.fcout1, self.fcout2),
            nn.BatchNorm1d(self.fcout2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.head = nn.Linear(self.fcout2, self.fcout3)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)

        x = torch.flatten(x, start_dim=1)

        x = self.block3(x)
        x = self.block4(x)
        x = self.head(x)

        return x


In [10]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    correct = 0
    total = 0

    for batch, labels in loader:
        batch = batch.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model(batch)
        loss = criterion(outputs, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size

        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += batch_size

    return total_loss / total, correct / total


In [11]:
model = Network(n_ratio=5.0).to(cfg.device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay,
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.1,
    patience=10,
)

best_val_acc = 0.0
best_state_dict = None


In [12]:
for epoch in range(cfg.num_epochs):
    k, t = update_ede_params(model, epoch, cfg.num_epochs)

    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch, labels in train_loader:
        batch = batch.to(cfg.device, non_blocking=True)
        labels = labels.to(cfg.device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(batch)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss += loss.item() * batch_size

        predicted = outputs.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += batch_size

    train_loss = running_loss / total
    train_acc = correct / total

    val_loss, val_acc = evaluate(model, val_loader, criterion, cfg.device)
    scheduler.step(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_dict = copy.deepcopy(model.state_dict())

    print(
        f"Epoch [{epoch + 1}/{cfg.num_epochs}] "
        f"k={k:.4f} t={t:.4f} "
        f"Train loss: {train_loss:.4f} "
        f"Train acc: {train_acc:.4f} "
        f"Val loss: {val_loss:.4f} "
        f"Val acc: {val_acc:.4f}"
    )

print(f"Best Validation Accuracy: {best_val_acc:.4f}")


Epoch [1/100] k=10.0000 t=0.1000 Train loss: 1.9335 Train acc: 0.3018 Val loss: 1.7516 Val acc: 0.3800
Epoch [2/100] k=9.5455 t=0.1048 Train loss: 1.7912 Train acc: 0.3593 Val loss: 1.6607 Val acc: 0.4060
Epoch [3/100] k=9.1116 t=0.1097 Train loss: 1.7481 Train acc: 0.3711 Val loss: 1.9295 Val acc: 0.3100
Epoch [4/100] k=8.6975 t=0.1150 Train loss: 1.7199 Train acc: 0.3828 Val loss: 1.7016 Val acc: 0.4120
Epoch [5/100] k=8.3022 t=0.1205 Train loss: 1.6936 Train acc: 0.3936 Val loss: 1.6984 Val acc: 0.3980
Epoch [6/100] k=7.9248 t=0.1262 Train loss: 1.6743 Train acc: 0.3993 Val loss: 1.7922 Val acc: 0.4040
Epoch [7/100] k=7.5646 t=0.1322 Train loss: 1.6469 Train acc: 0.4087 Val loss: 1.6245 Val acc: 0.4100
Epoch [8/100] k=7.2208 t=0.1385 Train loss: 1.6296 Train acc: 0.4164 Val loss: 1.5865 Val acc: 0.4220
Epoch [9/100] k=6.8926 t=0.1451 Train loss: 1.5955 Train acc: 0.4272 Val loss: 1.5832 Val acc: 0.4480
Epoch [10/100] k=6.5793 t=0.1520 Train loss: 1.5603 Train acc: 0.4402 Val loss: 1

In [13]:
if best_state_dict is not None:
    model.load_state_dict(best_state_dict)

clean_state_dict = copy.deepcopy(model.state_dict())

clean_test_loss, clean_test_acc = evaluate(model, test_loader, criterion, cfg.device)
print(
    f"Clean Test Loss: {clean_test_loss:.4f} "
    f"Clean Test Acc: {clean_test_acc:.4f}"
)


Clean Test Loss: 0.9400 Clean Test Acc: 0.6689


In [14]:
def set_ir_bit_flip_prob(model, p, base_seed=1234):
    layer_idx = 0

    for layer in model.modules():
        if isinstance(layer, (IRConv2d, IRLinear)):
            layer.perturb_prob = float(p)
            layer.perturb_seed = int(base_seed + layer_idx)
            layer_idx += 1


def reset_ir_bit_flips(model):
    for layer in model.modules():
        if isinstance(layer, (IRConv2d, IRLinear)):
            layer.perturb_prob = 0.0


In [15]:
device = next(model.parameters()).device
flip_percentages = list(range(0, 51, 5))

results = []

for pct in flip_percentages:
    p = pct / 100.0

    set_ir_bit_flip_prob(
        model=model,
        p=p,
        base_seed=1234,
    )

    test_loss, test_acc = evaluate(
        model=model,
        loader=test_loader,
        criterion=criterion,
        device=device,
    )

    results.append(
        {
            "flip_percent": pct,
            "flip_prob": p,
            "test_loss": test_loss,
            "test_acc": test_acc,
            "test_acc_percent": 100 * test_acc,
        }
    )

    print(
        f"Flip {pct:2d}% | "
        f"Test loss: {test_loss:.4f} | "
        f"Test acc: {100 * test_acc:.2f}%"
    )

reset_ir_bit_flips(model)


Flip  0% | Test loss: 0.9400 | Test acc: 66.89%
Flip  5% | Test loss: 1.0750 | Test acc: 61.35%
Flip 10% | Test loss: 1.3748 | Test acc: 50.67%
Flip 15% | Test loss: 1.8327 | Test acc: 35.10%
Flip 20% | Test loss: 1.9866 | Test acc: 26.45%
Flip 25% | Test loss: 2.3189 | Test acc: 16.92%
Flip 30% | Test loss: 2.4791 | Test acc: 13.71%
Flip 35% | Test loss: 2.3906 | Test acc: 13.42%
Flip 40% | Test loss: 2.5122 | Test acc: 11.19%
Flip 45% | Test loss: 2.5373 | Test acc: 9.46%
Flip 50% | Test loss: 2.4657 | Test acc: 10.09%
